In [2]:
import sys
import os
sys.path.append(os.path.abspath('..')) #for accesing folder

In [10]:
import pandas as pd
import numpy as np
import torch
import transformers
from transformers import BertTokenizer, BertForSequenceClassification, Trainer, TrainingArguments
from torch.utils.data import TensorDataset, DataLoader, RandomSampler, SequentialSampler
from torch.optim import AdamW
from datasets import Dataset
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix, classification_report, precision_score, recall_score, f1_score

import matplotlib.pyplot as plt
import seaborn as sns

#print(f"PyTorch version: {torch.__version__}")

In [5]:
#Loading model and tokenizer
import torch
from transformers import BertTokenizer, BertForSequenceClassification

device = torch.device('mps' if torch.backends.mps.is_available() else 'cpu')

tokenizer = BertTokenizer.from_pretrained('../models/bert_sqli_tokenizer_4_epochs_512_tokens')
model = BertForSequenceClassification.from_pretrained('../models/bert_sqli_model_4_epochs_512_tokens').to(device)
model.eval()
print(f"Model loaded on device: {device}")


Model loaded on device: mps


In [11]:
def predict(input_text, model,tokenizer, device):
    user_input=[input_text]
    user_encoding = tokenizer(user_input, truncation=True, padding=True, return_tensors='pt')
    user_dataset = TensorDataset(user_encoding['input_ids'], user_encoding['attention_mask'])
    user_loader=DataLoader(user_dataset, batch_size=1, shuffle=False)

    model.eval()
    with torch.no_grad():
        for batch in user_loader:
            input_ids, attention_mask = [t.to(device) for t in batch]
            outputs = model(input_ids, attention_mask=attention_mask) 
            logits = outputs.logits #these are the raw output scores for each class before applying softmax
            #predictions = torch.softmax(logits, dim=1) #Predicted label for input 'admin' OR '1'='1': tensor([[5.3014e-05, 9.9995e-01]], device='mps:0')
            predictions = torch.softmax(logits, dim=1)[0][1] #Predicted label for input 'admin' OR '1'='1': tensor(9.9995e-01), device='mps:0') - we want SQLi class
            precent_prediction = torch.softmax(logits, dim=1)[0][1].item() *100

    predicted_labels = 1 if (predictions.cpu().numpy() > 0.2).astype(int) else 0 #1 will be if the predicted probability for class 1 (SQLi) is greater than 0.2, otherwise 0
    return predicted_labels, precent_prediction #returning result predicted probability and percentage

In [8]:
print("Syntax is important, especially when using ' (single) or \"\" (double) quotes")

text="admin' OR '1'='1"
texts=["admin' OR '1'='1", 
       "SELECT * FROM users WHERE username = 'admin' --", 
       "SELECT * FROM users WHERE username = 'admin' AND password = 'password'", 
       "SELECT * FROM users WHERE username = 'admin' AND password = 'password' OR '1'='1'", 
       "SELECT * FROM users WHERE username = 'admin' AND password = 'password' OR '1'='1' --", 
       "SELECT * FROM users WHERE username = 'admin' AND password = 'password' OR '1'='1' #", 
       "SELECT * FROM users WHERE username = 'admin' AND password = 'password' OR '1'='1';", 
       "SELECT * FROM users WHERE username = 'admin' AND password = 'password' OR 1=1", 
       "SELECT * FROM users WHERE username = 'admin' AND password = 'password' OR 1=1 --", 
       "SELECT * FROM users WHERE username = 'admin' AND password = 'password' OR 1=1 #", 
       "SELECT * FROM users WHERE username = 'admin' AND password = 'password' OR 1=1;", 
       "SELECT * FROM users WHERE username = 'admin'; --", 
       "SELECT * FROM users WHERE username = 'admin'; #", 
       "SELECT * FROM users WHERE username = 'admin';", 
       "admin' ;"]

predicted_label, precent_prediction = predict(text, model, tokenizer, device)
print(f"Predicted label for input \" {text} \" : is {predicted_label}, with confidence of {precent_prediction:.4f}%")

for text in texts:
    predicted_label, precent_prediction = predict(text, model, tokenizer, device)
    print(f"Predicted label for input \" {text} \" : is {predicted_label}, with confidence of {precent_prediction:.4f}%")

Syntax is important, especially when using ' (single) or "" (double) quotes
Predicted label for input " admin' OR '1'='1 " : is 1, with confidence of 99.9947%
Predicted label for input " admin' OR '1'='1 " : is 1, with confidence of 99.9947%
Predicted label for input " SELECT * FROM users WHERE username = 'admin' -- " : is 0, with confidence of 13.5733%
Predicted label for input " SELECT * FROM users WHERE username = 'admin' AND password = 'password' " : is 0, with confidence of 0.2756%
Predicted label for input " SELECT * FROM users WHERE username = 'admin' AND password = 'password' OR '1'='1' " : is 1, with confidence of 99.8733%
Predicted label for input " SELECT * FROM users WHERE username = 'admin' AND password = 'password' OR '1'='1' -- " : is 1, with confidence of 99.9856%
Predicted label for input " SELECT * FROM users WHERE username = 'admin' AND password = 'password' OR '1'='1' # " : is 1, with confidence of 99.9300%
Predicted label for input " SELECT * FROM users WHERE usern

In [9]:
#Intersetd ones:
#1. Predicted label for input 'SELECT * FROM users WHERE username = 'admin';': is 0, with confidence of 0.0302%
#2. Predicted label for input 'admin' ;': is 1, with confidence of 52.6998%

In [55]:
#Unloading model and freeing up GPU memory
model.cpu()
del model
del tokenizer
import gc

gc.collect()
torch.mps.empty_cache()
print("Model unloaded!")

Model unloaded!
